In [ ]:
import sys, os
import pandas as pd
import numpy as np

# Adjust path to import utils from the repository root
sys.path.insert(0, os.path.abspath('..'))

from utils.config import Config, set_seed
from utils.data import load_close_panel, log_returns
from utils.features import make_features
from utils.signals import signal_set_a, signal_set_b, signal_set_c

# Initialize config and random seed for reproducibility
cfg = Config()
set_seed(42)

In [ ]:
# Load the cleaned panel data (prices and ETF holding prices)
# (Ensure load_close_panel arguments match your exact implementation if it requires the path)
prices, holding_prices = load_close_panel() 
returns = log_returns(prices)

# Generate base rolling features from the pipeline
pipeline_feats = make_features(prices, returns, holding_prices, cfg.roll_windows, cfg.past_window)

# Generate signal sets A, B, and C
sig_a = signal_set_a(prices, returns)
sig_b = signal_set_b(prices, returns)
sig_c = signal_set_c(prices, returns)

# Combine the individual signal DataFrames into one
pipeline_signals = sig_a.join([sig_b, sig_c], how='outer')

# Merge pipeline features and signals
pipeline_combined = pipeline_feats.join(pipeline_signals, how='inner').dropna()
print(f"Pipeline features & signals shape: {pipeline_combined.shape}")

In [ ]:
sp_500_df = pd.read_csv("../data/raw/processed_sp_500_macro_data.csv", index_col='Date', parse_dates=True)

# Separate the target column from the baseline features to prevent look-ahead bias
target_col = 'Target_Return_5D'
if target_col in sp_500_df.columns:
    sp500_features = sp_500_df.drop(columns=[target_col])
    sp500_target = sp_500_df[[target_col]]
else:
    sp500_features = sp_500_df
    sp500_target = None

print(f"S&P 500 macro features shape: {sp500_features.shape}")

In [ ]:
# Join the pipeline components with the macro features on the Date index
# An inner join ensures alignment and strips out mismatched dates between the datasets
final_features = pipeline_combined.join(sp500_features, how='inner').dropna()

# Align the target vector to the final trimmed dataset
if sp500_target is not None:
    final_target = sp500_target.reindex(final_features.index)

print(f"Final combined features shape: {final_features.shape}")

# Optional: Run PCA on the holding features here if dimension reduction is required prior to training
display(final_features.head())